# HateBERT Fine-tuning for PCL Detection
Fine-tunes HateBERT with weighted loss to handle class imbalance in the Don't Patronize Me! dataset.

## Imports

In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report
import ast
import os

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## Config

In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR       = 'datasets'
TSV_PATH       = os.path.join(DATA_DIR, 'dontpatronizeme_pcl.tsv')
TRAIN_PATH     = os.path.join(DATA_DIR, 'train_semeval_parids-labels.csv')
DEV_PATH       = os.path.join(DATA_DIR, 'dev_semeval_parids-labels.csv')

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME     = 'GroNLP/hateBERT'
MAX_LEN        = 128     # covers >95% of paragraphs based on EDA (mean=48, median=42)
BATCH_SIZE     = 32
EPOCHS         = 10
LEARNING_RATE  = 1e-5
WARMUP_RATIO   = 0.1     # 10% of steps used for warmup



## Load & Prepare Data

In [11]:
# Load main tsv
df = pd.read_csv(
    TSV_PATH,
    sep='\t',
    header=None,
    names=['par_id', 'art_id', 'keyword', 'country', 'text', 'label'],
    quoting=3,
    engine='python',
    skiprows=4
)

# Binary label: {0,1} -> 0 (No PCL), {2,3,4} -> 1 (PCL)
df['binary_label'] = df['label'].apply(lambda x: 0 if x in [0, 1] else 1)

# Load split files — we only need par_id from these
train_ids = pd.read_csv(TRAIN_PATH)['par_id'].astype(int).tolist()
dev_ids   = pd.read_csv(DEV_PATH)['par_id'].astype(int).tolist()

#Make sure par_id from tsv is also int
df['par_id'] = df['par_id'].astype(int)

# Filter to train and dev splits
train_df = df[df['par_id'].isin(train_ids)].reset_index(drop=True)
dev_df   = df[df['par_id'].isin(dev_ids)].reset_index(drop=True)

print(f'Train size : {len(train_df)} | PCL: {train_df["binary_label"].sum()} ({100*train_df["binary_label"].mean():.1f}%)')
print(f'Dev size   : {len(dev_df)}   | PCL: {dev_df["binary_label"].sum()} ({100*dev_df["binary_label"].mean():.1f}%)')


Train size : 8375 | PCL: 794 (9.5%)
Dev size   : 2094   | PCL: 199 (9.5%)


## Dataset & DataLoader

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PCLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'      : encoding['input_ids'].squeeze(),
            'attention_mask' : encoding['attention_mask'].squeeze(),
            'label'          : torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = PCLDataset(train_df['text'].tolist(), train_df['binary_label'].tolist(), tokenizer, MAX_LEN)
dev_dataset   = PCLDataset(dev_df['text'].tolist(),   dev_df['binary_label'].tolist(),   tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(SEED))
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)} | Dev batches: {len(dev_loader)}')

total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)
print(f'Total steps: {total_steps} | Warmup steps: {warmup_steps}')

Train batches: 262 | Dev batches: 66
Total steps: 2620 | Warmup steps: 262


## Training Loop

In [5]:
def train_epoch(model, loader, optimiser, scheduler, loss_fn, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimiser.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = loss_fn(outputs.logits, labels)
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimiser.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1       = f1_score(all_labels, all_preds, pos_label=1)
    return avg_loss, f1


def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs    = model(input_ids=input_ids, attention_mask=attention_mask)
            loss       = loss_fn(outputs.logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1       = f1_score(all_labels, all_preds, pos_label=1)
    return avg_loss, f1, all_preds, all_labels

## 1. HateBERT alone

In [23]:
# model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
# torch.save(model.state_dict(), 'hatebert_pretrained.pt') 

In [24]:

# Initialise model and optimiser
model_hatebert = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model_hatebert = model_hatebert.to(device)

# Standard cross-entropy — no class weights
loss_fn_hatebert = nn.CrossEntropyLoss()

optimiser_hatebert = AdamW(model_hatebert.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler_hatebert = get_linear_schedule_with_warmup(optimiser_hatebert, warmup_steps, total_steps)

best_f1 = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_f1 = train_epoch(model_hatebert, train_loader, optimiser_hatebert, scheduler_hatebert, loss_fn_hatebert, device)
    dev_loss, dev_f1, _, _ = evaluate(model_hatebert, dev_loader, loss_fn_hatebert, device)

    print(f'Epoch {epoch}/{EPOCHS}')
    print(f'  Train -> Loss: {train_loss:.4f} | F1: {train_f1:.4f}')
    print(f'  Dev   -> Loss: {dev_loss:.4f}   | F1: {dev_f1:.4f}')

    if dev_f1 > best_f1:
        best_f1 = dev_f1
        torch.save(model_hatebert.state_dict(), 'best_hatebert.pt')
        print(f'  >> New best model saved (F1: {best_f1:.4f})')

print(f'\nBest dev F1 (HateBERT alone): {best_f1:.4f}')

# Final evaluation
model_hatebert.load_state_dict(torch.load('best_hatebert.pt', map_location=device))
_, dev_f1, preds, labels = evaluate(model_hatebert, dev_loader, loss_fn_hatebert, device)

print('\n=== HateBERT alone===')
print(f'F1 (PCL class): {dev_f1:.4f}')
print()
print(classification_report(labels, preds, target_names=['No PCL', 'PCL']))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total steps: 2620 | Warmup steps: 262
Epoch 1/10
  Train -> Loss: 0.3857 | F1: 0.1014
  Dev   -> Loss: 0.2444   | F1: 0.1351
  >> New best model saved (F1: 0.1351)
Epoch 2/10
  Train -> Loss: 0.2263 | F1: 0.3274
  Dev   -> Loss: 0.2178   | F1: 0.4431
  >> New best model saved (F1: 0.4431)
Epoch 3/10
  Train -> Loss: 0.1690 | F1: 0.5824
  Dev   -> Loss: 0.2067   | F1: 0.4832
  >> New best model saved (F1: 0.4832)
Epoch 4/10
  Train -> Loss: 0.1143 | F1: 0.7543
  Dev   -> Loss: 0.2600   | F1: 0.4721
Epoch 5/10
  Train -> Loss: 0.0727 | F1: 0.8770
  Dev   -> Loss: 0.3078   | F1: 0.5381
  >> New best model saved (F1: 0.5381)
Epoch 6/10
  Train -> Loss: 0.0508 | F1: 0.9164
  Dev   -> Loss: 0.3584   | F1: 0.5167
Epoch 7/10
  Train -> Loss: 0.0311 | F1: 0.9499
  Dev   -> Loss: 0.4103   | F1: 0.4800
Epoch 8/10
  Train -> Loss: 0.0206 | F1: 0.9707
  Dev   -> Loss: 0.4413   | F1: 0.5326
Epoch 9/10
  Train -> Loss: 0.0146 | F1: 0.9785
  Dev   -> Loss: 0.4544   | F1: 0.5172
Epoch 10/10
  Train -> 

## 2. With Weighted Loss

### Compute Class Weights

In [8]:
# Weight each class inversely proportional to its frequency
n_samples  = len(train_df)
n_pcl      = train_df['binary_label'].sum()
n_no_pcl   = n_samples - n_pcl

weight_no_pcl = n_samples / (2 * n_no_pcl)
weight_pcl    = n_samples / (2 * n_pcl)

class_weights = torch.tensor([weight_no_pcl, weight_pcl], dtype=torch.float).to(device)
print(f'Class weights -> No PCL: {weight_no_pcl:.4f} | PCL: {weight_pcl:.4f}')

Class weights -> No PCL: 0.5524 | PCL: 5.2739


### Training

In [27]:
model_weightedl = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model_weightedl = model_weightedl.to(device)

# Weighted cross-entropy loss — penalises misclassifying PCL more heavily
loss_fn_weightedl = nn.CrossEntropyLoss(weight=class_weights)

optimiser_weightedl = AdamW(model_weightedl.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler_weightedl     = get_linear_schedule_with_warmup(optimiser_weightedl, warmup_steps, total_steps)

best_f1 = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_f1 = train_epoch(model_weightedl, train_loader, optimiser_weightedl, scheduler_weightedl, loss_fn_weightedl, device)
    dev_loss, dev_f1, _, _ = evaluate(model_weightedl, dev_loader, loss_fn_weightedl, device)

    print(f'Epoch {epoch}/{EPOCHS}')
    print(f'  Train -> Loss: {train_loss:.4f} | F1: {train_f1:.4f}')
    print(f'  Dev   -> Loss: {dev_loss:.4f}   | F1: {dev_f1:.4f}')

    # Save best model based on dev F1
    if dev_f1 > best_f1:
        best_f1 = dev_f1
        torch.save(model_weightedl.state_dict(), 'best_weightedl.pt')
        print(f'  >> New best model saved (F1: {best_f1:.4f})')

print(f'\nBest dev F1 (with weighted loss): {best_f1:.4f}')

model_weightedl.load_state_dict(torch.load('best_weightedl.pt', map_location=device))
_, dev_f1, preds, labels = evaluate(model_weightedl, dev_loader, loss_fn_weightedl, device)

print('=== Final Dev Set Results with weighted loss ===')
print(f'F1 (PCL class): {dev_f1:.4f}')
print()
print(classification_report(labels, preds, target_names=['No PCL', 'PCL']))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/10
  Train -> Loss: 0.6421 | F1: 0.2333
  Dev   -> Loss: 0.4798   | F1: 0.3815
  >> New best model saved (F1: 0.3815)
Epoch 2/10
  Train -> Loss: 0.4485 | F1: 0.4445
  Dev   -> Loss: 0.3590   | F1: 0.4877
  >> New best model saved (F1: 0.4877)
Epoch 3/10
  Train -> Loss: 0.3248 | F1: 0.5844
  Dev   -> Loss: 0.3560   | F1: 0.5040
  >> New best model saved (F1: 0.5040)
Epoch 4/10
  Train -> Loss: 0.2298 | F1: 0.7216
  Dev   -> Loss: 0.3658   | F1: 0.5232
  >> New best model saved (F1: 0.5232)
Epoch 5/10
  Train -> Loss: 0.1578 | F1: 0.8350
  Dev   -> Loss: 0.4489   | F1: 0.5506
  >> New best model saved (F1: 0.5506)
Epoch 6/10
  Train -> Loss: 0.1059 | F1: 0.8974
  Dev   -> Loss: 0.5176   | F1: 0.5309
Epoch 7/10
  Train -> Loss: 0.0640 | F1: 0.9389
  Dev   -> Loss: 0.6245   | F1: 0.5083
Epoch 8/10
  Train -> Loss: 0.0477 | F1: 0.9545
  Dev   -> Loss: 0.6586   | F1: 0.5280
Epoch 9/10
  Train -> Loss: 0.0284 | F1: 0.9716
  Dev   -> Loss: 0.6992   | F1: 0.5252
Epoch 10/10
  Train ->

## 3. Increasing Dropout

In [20]:
model_dropout = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model_dropout.classifier.dropout = nn.Dropout(p=0.3)
model_dropout = model_dropout.to(device)

loss_fn_dropout = nn.CrossEntropyLoss(weight=class_weights)

optimiser_dropout = AdamW(model_dropout.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler_dropout = get_linear_schedule_with_warmup(optimiser_dropout, warmup_steps, total_steps)

# Training loop
best_f1_dropout = 0
for epoch in range(1, EPOCHS + 1):
    train_loss, train_f1 = train_epoch(model_dropout, train_loader, optimiser_dropout, scheduler_dropout, loss_fn_dropout, device)
    dev_loss, dev_f1, _, _ = evaluate(model_dropout, dev_loader, loss_fn_dropout, device)

    print(f'Epoch {epoch}/{EPOCHS}')
    print(f'  Train -> Loss: {train_loss:.4f} | F1: {train_f1:.4f}')
    print(f'  Dev   -> Loss: {dev_loss:.4f}   | F1: {dev_f1:.4f}')

    if dev_f1 > best_f1_dropout:
        best_f1_dropout = dev_f1
        torch.save(model_dropout.state_dict(), 'best_dropout.pt')
        print(f'  >> New best model saved (F1: {best_f1_dropout:.4f})')

print(f'\nBest dev F1 (with higher dropout): {best_f1_dropout:.4f}')

# Final evaluation
model_dropout.load_state_dict(torch.load('best_dropout.pt', map_location=device))
_, dev_f1, preds, labels = evaluate(model_dropout, dev_loader, loss_fn_dropout, device)

print('\n=== Final Dev Set Results with higher dropout ===')
print(f'F1 (PCL class): {dev_f1:.4f}')
print()
print(classification_report(labels, preds, target_names=['No PCL', 'PCL']))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/10
  Train -> Loss: 0.6403 | F1: 0.2354
  Dev   -> Loss: 0.4817   | F1: 0.3847
  >> New best model saved (F1: 0.3847)
Epoch 2/10
  Train -> Loss: 0.4513 | F1: 0.4368
  Dev   -> Loss: 0.3641   | F1: 0.4821
  >> New best model saved (F1: 0.4821)
Epoch 3/10
  Train -> Loss: 0.3103 | F1: 0.5933
  Dev   -> Loss: 0.3760   | F1: 0.5051
  >> New best model saved (F1: 0.5051)
Epoch 4/10
  Train -> Loss: 0.2149 | F1: 0.7458
  Dev   -> Loss: 0.3900   | F1: 0.5497
  >> New best model saved (F1: 0.5497)
Epoch 5/10
  Train -> Loss: 0.1413 | F1: 0.8432
  Dev   -> Loss: 0.4884   | F1: 0.5515
  >> New best model saved (F1: 0.5515)
Epoch 6/10
  Train -> Loss: 0.0932 | F1: 0.9087
  Dev   -> Loss: 0.5701   | F1: 0.5459
Epoch 7/10
  Train -> Loss: 0.0640 | F1: 0.9451
  Dev   -> Loss: 0.6117   | F1: 0.5658
  >> New best model saved (F1: 0.5658)
Epoch 8/10
  Train -> Loss: 0.0446 | F1: 0.9558
  Dev   -> Loss: 0.6745   | F1: 0.5156
Epoch 9/10
  Train -> Loss: 0.0389 | F1: 0.9684
  Dev   -> Loss: 0.681

## 4. With Keyword Aware Input

### Add keyword to input

In [23]:
# Prepend the keyword to each text
train_texts_keyword = [
    f"{kw} {txt}" for kw, txt in zip(train_df['keyword'], train_df['text'])
]

dev_texts_keyword = [
    f"{kw} {txt}" for kw, txt in zip(dev_df['keyword'], dev_df['text'])
]

train_dataset_keyword = PCLDataset(
    train_texts_keyword,
    train_df['binary_label'].tolist(),
    tokenizer,
    MAX_LEN
)

dev_dataset_keyword = PCLDataset(
    dev_texts_keyword,
    dev_df['binary_label'].tolist(),
    tokenizer,
    MAX_LEN
)

train_loader_keyword = DataLoader(
    train_dataset_keyword,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED)
)

dev_loader_keyword = DataLoader(
    dev_dataset_keyword,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(f'Train batches (keyword): {len(train_loader_keyword)} | Dev batches (keyword): {len(dev_loader_keyword)}')

Train batches (keyword): 262 | Dev batches (keyword): 66


In [24]:
model_keyword = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model_keyword.classifier.dropout = nn.Dropout(p=0.3)
model_keyword = model_keyword.to(device)

loss_fn_keyword = nn.CrossEntropyLoss(weight=class_weights)

optimiser_keyword = AdamW(model_keyword.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler_keyword = get_linear_schedule_with_warmup(optimiser_keyword, warmup_steps, total_steps)

best_f1 = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_f1 = train_epoch(model_keyword, train_loader_keyword, optimiser_keyword, scheduler_keyword, loss_fn_keyword, device)
    dev_loss, dev_f1, _, _ = evaluate(model_keyword, dev_loader_keyword, loss_fn_keyword, device)

    print(f'Epoch {epoch}/{EPOCHS}')
    print(f'  Train -> Loss: {train_loss:.4f} | F1: {train_f1:.4f}')
    print(f'  Dev   -> Loss: {dev_loss:.4f}   | F1: {dev_f1:.4f}')

    # Save best model based on dev F1
    if dev_f1 > best_f1:
        best_f1 = dev_f1
        torch.save(model_keyword.state_dict(), 'best_keyword.pt')
        print(f'  >> New best model saved (F1: {best_f1:.4f})')

print(f'\nBest dev F1 (with keyword): {best_f1:.4f}')

model_keyword.load_state_dict(torch.load('best_keyword.pt', map_location=device))
_, dev_f1, preds, labels = evaluate(model_keyword, dev_loader_keyword, loss_fn_keyword, device)

print('=== Final Dev Set Results with keywords ===')
print(f'F1 (PCL class): {dev_f1:.4f}')
print()
print(classification_report(labels, preds, target_names=['No PCL', 'PCL']))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/10
  Train -> Loss: 0.6530 | F1: 0.2400
  Dev   -> Loss: 0.5623   | F1: 0.3458
  >> New best model saved (F1: 0.3458)
Epoch 2/10
  Train -> Loss: 0.4547 | F1: 0.4474
  Dev   -> Loss: 0.4210   | F1: 0.4582
  >> New best model saved (F1: 0.4582)
Epoch 3/10
  Train -> Loss: 0.3324 | F1: 0.5912
  Dev   -> Loss: 0.3719   | F1: 0.4946
  >> New best model saved (F1: 0.4946)
Epoch 4/10
  Train -> Loss: 0.2241 | F1: 0.7242
  Dev   -> Loss: 0.3328   | F1: 0.5505
  >> New best model saved (F1: 0.5505)
Epoch 5/10
  Train -> Loss: 0.1570 | F1: 0.8285
  Dev   -> Loss: 0.4040   | F1: 0.5687
  >> New best model saved (F1: 0.5687)
Epoch 6/10
  Train -> Loss: 0.0973 | F1: 0.8920
  Dev   -> Loss: 0.4821   | F1: 0.5678
Epoch 7/10
  Train -> Loss: 0.0723 | F1: 0.9290
  Dev   -> Loss: 0.5327   | F1: 0.5488
Epoch 8/10
  Train -> Loss: 0.0449 | F1: 0.9569
  Dev   -> Loss: 0.5832   | F1: 0.5305
Epoch 9/10
  Train -> Loss: 0.0296 | F1: 0.9709
  Dev   -> Loss: 0.5888   | F1: 0.5627
Epoch 10/10
  Train ->

## Final Evaluation on Dev Set

In [12]:
# Load best checkpoint and evaluate
model.load_state_dict(torch.load('best_keyword.pt', map_location=device))
_, dev_f1, preds, labels = evaluate(model, dev_loader, loss_fn, device)

print('=== Final Dev Set Results===')
print(f'F1 (PCL class): {dev_f1:.4f}')
print()
print(classification_report(labels, preds, target_names=['No PCL', 'PCL']))

=== Final Dev Set Results with weighted loss ===
F1 (PCL class): 0.5560

              precision    recall  f1-score   support

      No PCL       0.96      0.93      0.94      1895
         PCL       0.49      0.65      0.56       199

    accuracy                           0.90      2094
   macro avg       0.72      0.79      0.75      2094
weighted avg       0.92      0.90      0.91      2094

